In [8]:
import re
import torch
import faiss
import random
import requests
import evaluate
import numpy as np
from tqdm import tqdm
from bs4 import BeautifulSoup
from collections import Counter
from datasets import load_dataset
from transformers import LogitsProcessor
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [10]:
bnb_config = BitsAndBytesConfig(load_in_4bit = True,
                                bnb_4bit_compute_dtype = torch.float16,
                                bnb_4bit_use_double_quant = True,
                                bnb_4bit_quant_type = "nf4",
                                llm_int8_enable_fp32_cpu_offload = True)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2",
                                            quantization_config = bnb_config,    
                                            device_map="auto",
                                            torch_dtype=torch.float16)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [12]:

dataset = load_dataset('microsoft/ms_marco', 'v2.1')
for i in dataset['train'][0]:
    print(i, " ---- ", dataset['train'][0][i])

def clean_text(text):
    text = ' '.join(text.strip().split())
    text = re.sub(pattern='[^a-zA-Z0-9\\s,.:?!;]', repl='', string=text)
    return text


def print_in_chunks(text, chunk_size=10, skip_n_words=0):
    words = text.strip().split()
    chunks = []
    for i in range(skip_n_words, len(words), chunk_size):
        chunks.append(' '.join(words[i:i+chunk_size]))
    return '\n'.join(chunks)

def text_before_last_full_stop(text):
    idx = text.rfind(".")
    if idx == -1:
        return ""
    return text[:idx + 1]

def response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = model.generate(**inputs,
                                max_new_tokens=80,
                                temperature=0.7,
                                do_sample=True,
                                top_p=0.9,
                                eos_token_id=tokenizer.eos_token_id,
                                pad_token_id=tokenizer.eos_token_id)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        words = generated_text.strip().split()
        prompt_len = len(prompt.strip().split())
        answer_words = words[prompt_len:]
        answer = ' '.join(answer_words)
        answer = text_before_last_full_stop(answer)
        return answer


answers  ----  ['The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.']
passages  ----  {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this 

In [13]:

info = dict()
for i in tqdm(range(1)):
    info[i] = {"query": clean_text(dataset['train'][i]['query']),
               "true_answer": clean_text(dataset['train'][i]['answers'][0]),
               "before_rag_answer": response(prompt=f"Answer the following question in about {len(clean_text(dataset['train'][i]['answers'][0]).strip().split())} words : " + dataset['train'][i]['query'])
               }

print(print_in_chunks(text=info[0]['query']), "\n")
print(print_in_chunks(text=info[0]['true_answer']), "\n")
print(print_in_chunks(text=info[0]['before_rag_answer']), "\n")

corpus = dict()
p_id = 0
for i in tqdm(range(1)):
    is_selected_list = dataset['train'][i]['passages']['is_selected']
    passage_text = dataset['train'][i]['passages']['passage_text']
    for j, flag in enumerate(is_selected_list):
        if flag == 1:
            corpus[p_id] = passage_text[j]
            p_id += 1

100%|██████████| 1/1 [00:10<00:00, 10.28s/it]


what was the immediate impact of the success of the
manhattan project? 

The immediate impact of the success of the manhattan project
was the only cloud hanging over the impressive achievement of
the atomic researchers and engineers is what their success truly
meant; hundreds of thousands of innocent lives obliterated. 

The success of the Manhattan Project led to the deployment
of nuclear weapons against Japan in August 1945, effectively ending
World War II. It also established the United States as
a nuclear superpower and initiated the arms race between the
US and the Soviet Union. The scientific discoveries and technological
developments of the project also paved the way for future
nuclear research and applications. 



100%|██████████| 1/1 [00:00<?, ?it/s]


In [14]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
corpus_ids = list(corpus.keys())
corpus_texts = list(corpus.values())
corpus_embedding = embedder.encode(sentences=corpus_texts,
                                   convert_to_numpy=True,
                                   show_progress_bar=True)


d = corpus_embedding.shape[1]
index = faiss.IndexFlatL2(d)
index.add(corpus_embedding)


def retrieved_passage(query, k=1):
    query = clean_text(query)
    query_vector = embedder.encode(query, convert_to_numpy=True)
    D, I = index.search(np.array([query_vector]), k=k)
    passage = [corpus_texts[j] for j in I[0]]
    return passage

def after_rag_answer(query, k=1):
    retrieved = retrieved_passage(query=query)
    answer_len = None
    for i in range(1):
        if info[i]['query'] == query:
            answer_len = len(info[i]['true_answer'].split())
            break
    if answer_len is None:
        answer_len = 50
    prompt = f"Answer the following question in about {answer_len} words based on this given fact {retrieved} : {clean_text(query)}"
    answer = response(prompt=prompt)
    return answer

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
for i in tqdm(range(1)):
    rag_response = after_rag_answer(query=dataset['train'][i]['query'], k=1)
    info[i]['after_rag_answer'] = rag_response

print(print_in_chunks(text=info[0]['query']), "\n")
print(print_in_chunks(text=info[0]['true_answer']), "\n")
print(print_in_chunks(text=info[0]['before_rag_answer']), "\n")
print(print_in_chunks(text=info[0]['after_rag_answer']))

100%|██████████| 1/1 [00:08<00:00,  8.78s/it]

what was the immediate impact of the success of the
manhattan project? 

The immediate impact of the success of the manhattan project
was the only cloud hanging over the impressive achievement of
the atomic researchers and engineers is what their success truly
meant; hundreds of thousands of innocent lives obliterated. 

The success of the Manhattan Project led to the deployment
of nuclear weapons against Japan in August 1945, effectively ending
World War II. It also established the United States as
a nuclear superpower and initiated the arms race between the
US and the Soviet Union. The scientific discoveries and technological
developments of the project also paved the way for future
nuclear research and applications. 

The immediate impact of the success of the Manhattan Project
was the creation and use of the first atomic bombs,
resulting in the devastation of Hiroshima and Nagasaki and the
end of World War II. However, the moral implications of
this achievement, including the loss 

In [ ]:
api_key = "....................................." 

def internet_search(query):
    payload = {
        "api_key": api_key,
        "query": query,
        "max_results": 1
    }
    response = requests.post(
        "https://api.tavily.com/search",
        json=payload,
        headers={"Content-Type": "application/json"}
    )
    data = response.json()
    return data['results']

def scrape_page(url):
    try:
        html = requests.get(url, timeout=10).text
    except Exception:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "header", "footer", "nav", "aside"]):
        tag.extract()
    paragraphs = soup.find_all("p")
    text = " ".join(p.get_text().strip() for p in paragraphs)
    return text

def full_web_search(query):
    results = internet_search(query)
    pages = []
    for item in results:
        url = item["url"]
        title = item.get("title", "")
        text = scrape_page(url)
        pages.append({"title": title, "url": url, "content": text})
    return pages

def internet_RAG(query):
    answer = full_web_search(clean_text(query))
    if not answer:
        return "No web content retrieved."
    context = answer[0]['content']
    # Determine answer length (fallback to 50 if not found)
    answer_len = None
    for i in range(1):
        if dataset['train'][i]['query'] == query:
            answer_len = len(dataset['train'][i]['answers'][0].split())
            break
    if answer_len is None:
        answer_len = 50
    prompt = f"Answer the following question in about {answer_len} words based on this given fact {context} : {clean_text(query)}"
    result = response(prompt=prompt)
    return result

In [17]:
# Optional: test internet RAG on a different query
internet_rag_response = internet_RAG(query="How much is the cost of Mercedes Benz S class")
info[0]['internet_rag_answer'] = internet_rag_response

In [18]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def neural_rag_retrieve_score(query, k=1):
    """
    Retrieve top k passages and return them along with the cosine similarity
    between the query embedding and the top passage embedding.
    """
    query_clean = clean_text(query)
    query_vec = embedder.encode(query_clean, convert_to_numpy=True)
    D, I = index.search(np.array([query_vec]), k=k)
    passages = [corpus_texts[j] for j in I[0]]
    # Compute similarity with top retrieved passage
    if len(passages) > 0:
        passage_vec = embedder.encode(passages[0], convert_to_numpy=True)
        sim = cosine_similarity(query_vec, passage_vec)
    else:
        sim = 0.0
    return passages, sim

def generate_answer_from_context(query, context, answer_len=None):
    """
    Generate answer using the provided context (string or list).
    """
    if answer_len is None:
        # Try to infer from dataset (for the sample query)
        for i in range(1):
            if info[i]['query'] == query:
                answer_len = len(info[i]['true_answer'].split())
                break
        if answer_len is None:
            answer_len = 50
    if isinstance(context, list):
        context_str = " ".join(context)
    else:
        context_str = context
    prompt = f"Answer the following question in about {answer_len} words based on this given fact {context_str} : {clean_text(query)}"
    return response(prompt=prompt)

def answer_with_fallback(query, threshold=0.50):
    """
    Main routing function.
    Returns answer, source, and similarity score.
    """
    print(f"\n--- Processing query: {query} ---")
    # Step 1: Neural retrieval + similarity score
    retrieved_passages, sim_score = neural_rag_retrieve_score(query, k=1)
    print(f"Neural RAG similarity score: {sim_score:.4f}")

    if sim_score >= threshold:
        print(f"Score >= {threshold} -> using Neural RAG (no fallback)")
        source = "Neural RAG"
        answer = generate_answer_from_context(query, retrieved_passages)
    else:
        print(f"Score < {threshold} -> FALLBACK triggered. Switching to Internet RAG.")
        source = "Internet RAG"
        answer = internet_RAG(query)   # uses the original query

    print(f"Source used: {source}")
    print(f"Final answer (first 200 chars): {answer[:200]}...")
    return answer, source, sim_score

In [21]:
test_query = dataset['train'][0]['query']
final_ans, source_used, score = answer_with_fallback("what is the price of Mercedes Benz S class", threshold=0.50)
info[0]['routed_answer'] = final_ans
info[0]['routing_source'] = source_used
info[0]['neural_similarity_score'] = score

print("=" * 50)
print("QUERY:")
print("\nROUTED ANSWER (with fallback):")
print(print_in_chunks(info[0]['routed_answer']))
print(f"\nRouting source: {info[0]['routing_source']}")
print(f"Similarity score: {info[0]['neural_similarity_score']:.4f}")


--- Processing query: what is the price of Mercedes Benz S class ---
Neural RAG similarity score: -0.0983
Score < 0.5 -> FALLBACK triggered. Switching to Internet RAG.
Source used: Internet RAG
Final answer (first 200 chars): ? The Mercedes-Benz S-Class is a luxury vehicle with an exquisite design and advanced features. Pricing varies depending on specifications, with the base model starting around $93,000 and the top-of-t...
QUERY:

ROUTED ANSWER (with fallback):
? The Mercedes-Benz S-Class is a luxury vehicle with an
exquisite design and advanced features. Pricing varies depending on specifications,
with the base model starting around $93,000 and the top-of-the-line
model reaching over $180,000. Financing and leasing options are also
available.

Routing source: Internet RAG
Similarity score: -0.0983
